# Statistical Inference & Confidence Intervals

Companion notebook for the [Statistical Inference lesson](https://ml-viz-ruby.vercel.app/courses/probability-statistics/06-statistical-inference).

We make the abstractions concrete by **simulation**: watch the Central Limit Theorem turn a skewed
population into normal sample means, confirm the standard error shrinks like 1/√n, and check that a
95% confidence interval really does cover the truth ~95% of the time. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — The Central Limit Theorem in action

We draw from a strongly skewed (exponential) population. For each sample size n we take many samples,
record each sample's mean, and histogram them. The skew washes out and the means become bell-shaped
as n grows — regardless of the population's shape.

In [ ]:
pop_mean = 1.0          # exponential(scale=1) has mean 1 and SD 1
pop_sd = 1.0
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for ax, n in zip(axes, [1, 2, 10, 50]):
    means = rng.exponential(scale=1.0, size=(5000, n)).mean(axis=1)
    ax.hist(means, bins=40, color='#6366f1', density=True)
    ax.axvline(pop_mean, color='#fb7185', ls='--')
    ax.set_title(f'n = {n}'); ax.set_xlim(0, 3)
axes[0].set_ylabel('density')
fig.suptitle('Sampling distribution of the mean approaches normal as n grows')
plt.tight_layout(); plt.show()

## 2 — Standard error shrinks like 1/√n

The theoretical standard error is SE = σ/√n. We compare it to the *measured* spread of sample means
across many simulated samples — they should match closely.

In [ ]:
ns = np.array([1, 2, 5, 10, 25, 50, 100, 200])
theoretical = pop_sd / np.sqrt(ns)
measured = [rng.exponential(1.0, size=(4000, n)).mean(axis=1).std() for n in ns]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(ns, theoretical, 'o-', color='#2dd4bf', label='theoretical  σ/√n')
ax.plot(ns, measured, 's--', color='#fb7185', label='measured spread of means')
ax.set_xlabel('sample size n'); ax.set_ylabel('standard error')
ax.set_title('Standard error follows σ/√n: 4x the data halves the error')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"n=25  -> SE = {pop_sd/np.sqrt(25):.3f}")
print(f"n=100 -> SE = {pop_sd/np.sqrt(100):.3f}  (4x the data -> half the SE)")

## 3 — Do 95% confidence intervals really cover 95%?

We build a 95% CI (`x̄ ± 1.96·SE`) from many independent samples of a normal population and count how
often the interval contains the *true* mean. The coverage should land near 95% — that long-run
coverage is exactly what '95% confidence' means.

In [ ]:
true_mu, true_sigma, n, trials = 5.0, 2.0, 40, 10000
z_star = 1.96
covered = 0
for _ in range(trials):
    sample = rng.normal(true_mu, true_sigma, size=n)
    xbar = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(n)
    lo, hi = xbar - z_star * se, xbar + z_star * se
    if lo <= true_mu <= hi:
        covered += 1
print(f"empirical coverage of nominal 95% CI: {100*covered/trials:.1f}%  (target 95%)")

## ✏️ Your turn

**Exercise.** Implement `standard_error(sigma, n)` and `confidence_interval(xbar, sigma, n, z_star)`
returning the `(low, high)` tuple for a CI of the mean. From the lesson: `SE = σ/√n` and the interval
is `x̄ ± z*·SE`.

In [ ]:
def standard_error(sigma, n):
    # TODO(you): return the standard error of the mean
    return ...

def confidence_interval(xbar, sigma, n, z_star=1.96):
    # TODO(you): return (low, high) for x̄ ± z*·SE
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert standard_error(20, 100) == 2.0
assert abs(standard_error(20, 64) - 2.5) < 1e-9
lo, hi = confidence_interval(50, 20, 100)        # SE = 2, margin = 3.92
assert abs(lo - 46.08) < 1e-6 and abs(hi - 53.92) < 1e-6
print("\u2713 SE and confidence interval are correct")

<details>
<summary>Solution</summary>

```python
def standard_error(sigma, n):
    return sigma / np.sqrt(n)

def confidence_interval(xbar, sigma, n, z_star=1.96):
    se = standard_error(sigma, n)
    return (xbar - z_star * se, xbar + z_star * se)
```

The √n in the standard error is the central fact of inference: precision improves only as the square
root of the data, so each extra digit of accuracy costs disproportionately more samples.

</details>